In [2]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

import mlflow

pl.Config.set_tbl_cols(-1)       # Wyświetl wszystkie kolumny
pl.Config.set_tbl_rows(20)       # Ustaw limit wierszy (lub -1 dla wszystkich)
pl.Config.set_fmt_str_lengths(100) # Nie skracaj długich ciągów (np. SMILES)

/home/computer/Repositories/ml_chembl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


polars.config.Config

In [3]:
df = pl.read_parquet("processed_data/ChEMBL_processed.parquet")

In [4]:
print(df.columns)

['activity_id', 'molregno', 'canonical_smiles', 'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb', 'aromatic_rings', 'qed_weighted', 'standard_value', 'standard_units', 'standard_type', 'standard_relation', 'pchembl_value', 'target_chembl_id', 'target_name', 'confidence_score', 'pIC50']


In [5]:
radius = 2
n_bits = 2048
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

def get_scaffold_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

# Konwersja SMILES -> Fingerprint
def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,), dtype=np.float32)
    return mfpgen.GetFingerprintAsNumPy(mol).astype(np.float32)

# Architektura zgodna z wytycznymi
class MLPBaseline(nn.Module):
    def __init__(self, input_size=2048):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x)

class GNNBaseline(torch.nn.Module):
    def __init__(self, node_features):
        super().__init__()
        self.conv1 = GCNConv(node_features, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 64)
        self.lin1 = torch.nn.Linear(64, 32)
        self.lin2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x)

In [6]:
## Uczenie
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
optimizer = None
criterion = torch.nn.MSELoss()

def train_step(model, batch, optimizer, criterion, device, is_gnn=False):
    model.train()
    optimizer.zero_grad()

    if is_gnn:
        batch = batch.to(device)
        preds = model(batch)
        target = batch.y.view(-1, 1)
    else:
        x, y = batch
        preds = model(x.to(device))
        target = y.to(device).view(-1, 1)

    loss = criterion(preds, target)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()

In [7]:
def train_one_epoch(model, loader, optimizer, criterion, device, is_gnn=False):
    model.train()
    total_loss = 0
    for batch in loader:
        if is_gnn:
            batch = batch.to(device)
            target = batch.y.view(-1, 1)
            out = model(batch)
        else:
            x, y = batch
            x, target = x.to(device), y.to(device).view(-1, 1)
            out = model(x)

        loss = criterion(out, target)
        if torch.isnan(loss):
            print("Loss exploded to NaN! Stopping...")
            return float('nan')

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, is_gnn=False):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            if is_gnn:
                batch = batch.to(device)
                out = model(batch)
                target = batch.y.view(-1, 1)
            else:
                x, y = batch
                out = model(x.to(device))
                target = y.view(-1, 1)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())
    all_preds = np.concatenate(all_preds).ravel()
    all_targets = np.concatenate(all_targets).ravel()
    if np.isnan(all_preds).any():
        print(f"BŁĄD: Predykcje modelu zawierają NaN! (Pierwsze 5: {all_preds[:5]})")
    if np.isnan(all_targets).any():
        print("BŁĄD: Etykiety w zbiorze walidacyjnym zawierają NaN!")
    return r2_score(all_targets, all_preds)

In [8]:
def get_valid_fp(smiles):
    """Próbuje stworzyć molekułę i fingerprint. Zwraca None przy błędzie."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return mfpgen.GetFingerprintAsNumPy(mol).astype(np.float32)
    except Exception:
        return None

df_temp = df.with_columns(
    pl.col("canonical_smiles").map_elements(get_valid_fp, return_dtype=pl.Object).alias("fp")
)
df_clean = df_temp.filter(
    (pl.col("fp").is_not_null()) & 
    (pl.col("pIC50").is_not_null()) &
    (pl.col("pIC50").is_not_nan())
)
print(f"Liczba próbek po pełnym oczyszczeniu: {len(df_clean)}")
print(f"Odrzucono {len(df) - len(df_clean)} błędnych cząsteczek.")

[22:28:00] WARNING: not removing hydrogen atom without neighbors
[22:28:16] Can't kekulize mol.  Unkekulized atoms: 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77
[22:28:38] Explicit valence for atom # 1 P, 7, is greater than permitted
[22:28:47] Explicit valence for atom # 34 P, 7, is greater than permitted
[22:29:04] WARNING: not removing hydrogen atom without neighbors


Liczba próbek po pełnym oczyszczeniu: 561058
Odrzucono 4 błędnych cząsteczek.


In [9]:
def random_split_arrays(X, y, test_size=0.1, val_size=0.1, seed=42):
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=test_size + val_size, random_state=seed)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=(test_size / (test_size + val_size)), random_state=seed)
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def scaffold_split(df_fp: pl.DataFrame, test_size=0.1, val_size=0.1, seed=42):
    df_scaff = df_fp.with_columns(
        pl.col("canonical_smiles").map_elements(get_scaffold_smiles, return_dtype=pl.Utf8).alias("scaffold")
)
    df_scaff = df_scaff.filter(pl.col("scaffold").is_not_null())
    scaffolds = df_scaff["scaffold"].to_list()
    rng = np.random.default_rng(seed)
    unique_scaff = list(set(scaffolds))
    rng.shuffle(unique_scaff)
    n_test = max(1, int(len(unique_scaff) * test_size))
    n_val = max(1, int(len(unique_scaff) * val_size))
    test_scaff = set(unique_scaff[:n_test])
    val_scaff = set(unique_scaff[n_test:n_test + n_val])
    def assign_split(scaff):
        if scaff in test_scaff:
            return "test"
        if scaff in val_scaff:
            return "val"
        return "train"
    df_labeled = df_scaff.with_columns(
        pl.col("scaffold").map_elements(assign_split, return_dtype=pl.Utf8).alias("split")
)
    return (
        df_labeled.filter(pl.col("split") == "train"),
        df_labeled.filter(pl.col("split") == "val"),
        df_labeled.filter(pl.col("split") == "test"),
)

def build_mlp_loaders(df_fp: pl.DataFrame, split_type="random", batch_size=64, seed=42):
    if split_type == "scaffold":
        train_df, val_df, test_df = scaffold_split(df_fp, seed=seed)
        X_train = np.stack(train_df["fp"].to_list())
        y_train = train_df["pIC50"].to_numpy().astype(np.float32)
        X_val = np.stack(val_df["fp"].to_list())
        y_val = val_df["pIC50"].to_numpy().astype(np.float32)
        X_test = np.stack(test_df["fp"].to_list())
        y_test = test_df["pIC50"].to_numpy().astype(np.float32)
    else:
        X = np.stack(df_fp["fp"].to_list())
        y = df_fp["pIC50"].to_numpy().astype(np.float32)
        (X_train, y_train), (X_val, y_val), (X_test, y_test) = random_split_arrays(X, y, seed=seed)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)), batch_size=batch_size)
    return train_loader, val_loader, test_loader

In [10]:
def mol_to_graph(smiles: str, target: float):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_feats = []
    for atom in mol.GetAtoms():
        node_feats.append([
            atom.GetAtomicNum(),
            atom.GetTotalDegree(),
            atom.GetFormalCharge(),
            atom.GetTotalNumHs(),
            int(atom.GetIsAromatic()),
        ])
    x = torch.tensor(node_feats, dtype=torch.float)
    edge_index_list = []
    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        edge_index_list.extend([[u, v], [v, u]])
    if edge_index_list:
        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=torch.tensor([target], dtype=torch.float))

def build_gnn_loaders(df_fp: pl.DataFrame, split_type="random", batch_size=64, seed=42):
    if split_type == "scaffold":
        train_df, val_df, test_df = scaffold_split(df_fp, seed=seed)
    else:
        df_fp = df_fp.sample(fraction=1.0, seed=seed)
        n = len(df_fp)
        n_test = int(0.1 * n)
        n_val = int(0.1 * n)
        test_df = df_fp[:n_test]
        val_df = df_fp[n_test:n_test + n_val]
        train_df = df_fp[n_test + n_val:]
    def df_to_graphs(df_part):
        graphs = []
        for row in df_part.iter_rows(named=True):
            g = mol_to_graph(row["canonical_smiles"], float(row["pIC50"]))
            if g is not None:
                graphs.append(g)
        return graphs
    train_graphs = df_to_graphs(train_df)
    val_graphs = df_to_graphs(val_df)
    test_graphs = df_to_graphs(test_df)
    train_loader = GeoDataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = GeoDataLoader(val_graphs, batch_size=batch_size)
    test_loader = GeoDataLoader(test_graphs, batch_size=batch_size)
    return train_loader, val_loader, test_loader

In [11]:
## Główna pętla treningowa z MLflow
EPOCHS = 50
LR = 1e-4
BATCH_SIZE = 64
SPLIT_TYPE = "random"  # "random" lub "scaffold"
MODEL_TYPE = "MLP"      # "MLP" lub "GNN"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if MODEL_TYPE == "MLP":
    train_loader, val_loader, test_loader = build_mlp_loaders(df_clean, split_type=SPLIT_TYPE, batch_size=BATCH_SIZE)
    model = MLPBaseline().to(DEVICE)
    is_gnn_flag = False
elif MODEL_TYPE == "GNN":
    train_loader, val_loader, test_loader = build_gnn_loaders(df_clean, split_type=SPLIT_TYPE, batch_size=BATCH_SIZE)
    sample_graph = train_loader.dataset[0]
    model = GNNBaseline(sample_graph.num_node_features).to(DEVICE)
    is_gnn_flag = True
else:
    raise ValueError("MODEL_TYPE must be 'MLP' or 'GNN'")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.MSELoss()

with mlflow.start_run(run_name=f"Baseline_{MODEL_TYPE}_{SPLIT_TYPE}"):
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("model_type", MODEL_TYPE)
    mlflow.log_param("split_type", SPLIT_TYPE)
    
    for epoch in range(EPOCHS):
        avg_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, is_gnn=is_gnn_flag)
        r2_val = evaluate(model, val_loader, DEVICE, is_gnn=is_gnn_flag)
        mlflow.log_metric("mse_loss", avg_loss, step=epoch)
        mlflow.log_metric("r2_val", r2_val, step=epoch)
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss {avg_loss:.4f} | R2 Val: {r2_val:.4f}")
    # Ewaluacja testowa
    r2_test = evaluate(model, test_loader, DEVICE, is_gnn=is_gnn_flag)
    mlflow.log_metric("r2_test", r2_test)
    print(f"Test R2: {r2_test:.4f}")
    mlflow.pytorch.log_model(model, "model")

Epoch 0: Loss 1.8056 | R2 Val: 0.4388
Epoch 10: Loss 0.6560 | R2 Val: 0.5924
Epoch 20: Loss 0.4759 | R2 Val: 0.5935
Epoch 30: Loss 0.3850 | R2 Val: 0.5898
Epoch 40: Loss 0.3333 | R2 Val: 0.5862


2026/03/28 22:38:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 22:38:12 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Test R2: 0.5852


2026/03/28 22:38:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [12]:
fp_has_nan = any(np.isnan(fp).any() for fp in df_clean["fp"])
print(f"NaN w fingerprintach: {fp_has_nan}")

NaN w fingerprintach: False


In [14]:
# Narzędzia do pojedynczego treningu i kolekcji wyników
results_table = []


def train_and_score(model_type: str, split_type: str, epochs: int = 15, lr: float = 1e-4, batch_size: int = 64, log_mlflow: bool = False):
    """Trenuje wskazany model na zadanym splicie i zwraca metryki."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if model_type == "MLP":
        train_loader, val_loader, test_loader = build_mlp_loaders(df_clean, split_type=split_type, batch_size=batch_size)
        model = MLPBaseline().to(device)
        is_gnn_flag = False
    elif model_type == "GNN":
        train_loader, val_loader, test_loader = build_gnn_loaders(df_clean, split_type=split_type, batch_size=batch_size)
        sample_graph = train_loader.dataset[0]
        model = GNNBaseline(sample_graph.num_node_features).to(device)
        is_gnn_flag = True
    else:
        raise ValueError("model_type must be 'MLP' or 'GNN'")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()

    run_ctx = mlflow.start_run(run_name=f"{model_type}_{split_type}") if log_mlflow else None
    if log_mlflow:
        mlflow.log_params({"model_type": model_type, "split_type": split_type, "epochs": epochs, "lr": lr})

    for epoch in range(epochs):
        train_one_epoch(model, train_loader, optimizer, criterion, device, is_gnn=is_gnn_flag)

    r2_val = evaluate(model, val_loader, device, is_gnn=is_gnn_flag)
    r2_test = evaluate(model, test_loader, device, is_gnn=is_gnn_flag)

    if log_mlflow:
        mlflow.log_metric("r2_val", r2_val)
        mlflow.log_metric("r2_test", r2_test)
        mlflow.pytorch.log_model(model, "model")
        mlflow.end_run()

    result = {
        "model": model_type,
        "split": split_type,
        "epochs": epochs,
        "lr": lr,
        "r2_val": r2_val,
        "r2_test": r2_test,
    }
    results_table.append(result)
    print(f"{model_type} | {split_type}: val R2={r2_val:.3f}, test R2={r2_test:.3f}")
    return result

In [15]:
# MLP - random split (szybki baseline)
train_and_score(model_type="MLP", split_type="random", epochs=15, lr=1e-4, batch_size=64, log_mlflow=False)

MLP | random: val R2=0.597, test R2=0.595


{'model': 'MLP',
 'split': 'random',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.5973082780838013,
 'r2_test': 0.5954588651657104}

In [16]:
# MLP - scaffold split
train_and_score(model_type="MLP", split_type="scaffold", epochs=15, lr=1e-4, batch_size=64, log_mlflow=False)

[23:24:44] WARNING: not removing hydrogen atom without neighbors
[23:26:22] WARNING: not removing hydrogen atom without neighbors


MLP | scaffold: val R2=0.486, test R2=0.448


{'model': 'MLP',
 'split': 'scaffold',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.4862673282623291,
 'r2_test': 0.44808781147003174}

In [17]:
# GNN - random split
train_and_score(model_type="GNN", split_type="random", epochs=15, lr=1e-4, batch_size=64, log_mlflow=False)

[23:31:49] WARNING: not removing hydrogen atom without neighbors
[23:34:30] WARNING: not removing hydrogen atom without neighbors


GNN | random: val R2=0.089, test R2=0.088


{'model': 'GNN',
 'split': 'random',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.08889180421829224,
 'r2_test': 0.08756983280181885}

In [18]:
# GNN - scaffold split
train_and_score(model_type="GNN", split_type="scaffold", epochs=15, lr=1e-4, batch_size=64, log_mlflow=False)

[23:43:13] WARNING: not removing hydrogen atom without neighbors
[23:44:47] WARNING: not removing hydrogen atom without neighbors
[23:45:51] WARNING: not removing hydrogen atom without neighbors
[23:46:10] WARNING: not removing hydrogen atom without neighbors


GNN | scaffold: val R2=0.079, test R2=0.077


{'model': 'GNN',
 'split': 'scaffold',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.07880210876464844,
 'r2_test': 0.07722675800323486}

In [19]:
# Zestawienie wyników w tabeli
import pandas as pd

if not results_table:
    print("Brak zebranych wyników. Uruchom najpierw komórki treningowe.")
else:
    df_results = pd.DataFrame(results_table)
    display(df_results)
    print("Posortowane wg r2_test:")
    display(df_results.sort_values("r2_test", ascending=False))

,model,split,epochs,lr,r2_val,r2_test
0,MLP,random,15,0.0001,0.597308,0.595459
1,MLP,scaffold,15,0.0001,0.486267,0.448088
2,GNN,random,15,0.0001,0.088892,0.087570
3,GNN,scaffold,15,0.0001,0.078802,0.077227


Posortowane wg r2_test:


,model,split,epochs,lr,r2_val,r2_test
0,MLP,random,15,0.0001,0.597308,0.595459
1,MLP,scaffold,15,0.0001,0.486267,0.448088
2,GNN,random,15,0.0001,0.088892,0.087570
3,GNN,scaffold,15,0.0001,0.078802,0.077227
